# CKA-RL on MiniGrid DoorKey — Kaggle runner**Panel settings:** Accelerator = **GPU T4 x2**, Internet = **On**. Then`Save Version` → **Save & Run All (Commit)** and close the tab.The whole plan is only a few GPU-hours here, because a MiniGrid step costsorders of magnitude less than a MuJoCo step and the SAC update is whatdominates. One stage per commit is still the safest pattern.| order | STAGE | ARG | roughly | what it answers ||---|---|---|---|---|| 1 | `smoke` | — | ~3 min | does the pipeline run end to end || 2 | `pilot` | — | ~15 min | do the four tasks learn in 60k steps || 3 | `baselines` | — | ~15 min | from-scratch references for FT || 4 | `cond` | `1`…`4` | ~20-40 min each | one condition, all seeds || 5 | `report` | — | minutes | metrics + plots |Conditions: `1` baseline (CKA-RL), `2` distil_only, `3` weight_only,`4` combined.

## 1. Config

In [ ]:
REPO_URL    = "https://github.com/Yasamin-Rajabi/Continual-RL-Project.git"REPO_BRANCH = "Narges"REPO_SUBDIR = "minigrid"      # folder inside the repo; "" if the code is at the rootSTAGE = "smoke"               # smoke | pilot | baselines | cond | report | setup | sanityARG   = ""                    # for cond: "1".."4"BUDGET_HOURS = 10.5           # stop before the 12 h wall so the version still commitsimport os, pathlibWORK = pathlib.Path("/kaggle/working")CODE = pathlib.Path("/kaggle/temp/repo")   # scratch: not part of the saved outputprint("stage:", STAGE, ARG, "| budget:", BUDGET_HOURS, "h")

## 2. Pull the code

In [ ]:
import subprocess, shutilif CODE.exists():    shutil.rmtree(CODE)CODE.parent.mkdir(parents=True, exist_ok=True)subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(CODE)], check=True)PROJ = CODE / REPO_SUBDIR if REPO_SUBDIR else CODEassert (PROJ / "run_kaggle.sh").exists(), f"run_kaggle.sh not in {PROJ} - check REPO_SUBDIR"os.chdir(PROJ)print("code at", PROJ)print(subprocess.run(["git", "log", "-1", "--oneline"], cwd=CODE,                     capture_output=True, text=True).stdout.strip())

## 3. Resume from the previous session`Save & Run All` starts from an empty `/kaggle/working`. To continue, add theprevious version's output as an input (`+ Add Input` in the right-hand panel →`Your Work`); this cell copies it back so finished work is skipped.No-op on the first run.

In [ ]:
import globrestored = 0for src_root in sorted(glob.glob("/kaggle/input/*")):    for name in ("runs", "agents", "analysis", "analysis_scratch",                 "scratch_models", "plots", "logs"):        src = pathlib.Path(src_root) / name        if not src.is_dir():            continue        shutil.copytree(src, WORK / name, dirs_exist_ok=True)        n = sum(1 for _ in src.rglob("*"))        restored += n        print(f"restored {name} from {src_root} ({n} entries)")print("nothing to restore - first run" if restored == 0 else f"restored {restored} entries")

## 4. InstallMiniGrid is a normal PyPI package, so no source builds or pinned commits areneeded. The setup stage also runs a real CUDA kernel: `torch.cuda.is_available()`returns True on a P100 even though this wheel has no sm_60 kernels, and thefailure would otherwise only surface deep inside training.

In [ ]:
os.environ["KAGGLE_WORKING"] = str(WORK)!bash run_kaggle.sh setup

## 5. Sanity check (no GPU time)Builds every task and asserts a constant observation/action space across thesuite plus the presence of the `success` and `task_error` info keys.

In [ ]:
!bash run_kaggle.sh sanity

## 6. Run the stageExit code 124 means the budget was reached, not a failure: everything finishedso far is checkpointed and the next session resumes from it.

In [ ]:
import timesecs = int(BUDGET_HOURS * 3600)cmd = f"timeout --signal=INT {secs} bash run_kaggle.sh {STAGE} {ARG}".strip()print(cmd, flush=True)t0 = time.time()rc = subprocess.run(cmd, shell=True).returncodeelapsed = (time.time() - t0) / 3600print(f"\nstage={STAGE} {ARG} rc={rc} elapsed={elapsed:.2f} h")if rc == 124:    print("BUDGET REACHED - not a failure. Commit this version, add its output "          "as input, and run the same stage again.")elif rc != 0:    raise SystemExit(f"stage failed with code {rc} - see the log above")else:    print("stage complete.")

## 7. What was produced

In [ ]:
print(subprocess.run(f"du -sh {WORK}/* 2>/dev/null | sort -h", shell=True,                     capture_output=True, text=True).stdout)print("\nNext: Save Version -> Save & Run All (Commit), then add THIS version's "      "output as an input dataset before the next stage.")